# 1.- Drop-Medallion

Notebook de **reversión** del proyecto **Análisis de Movilidad Multimodal de NYC**.

El objetivo es eliminar las **tablas lógicas** registradas en Unity Catalog y sus **rutas físicas** en las capas Bronze, Silver y Golden.

> **Importante:** este notebook **NO elimina la capa Raw**, ni el catálogo, ni los schemas, ni las External Locations, ni el Storage Credential, ni el Access Connector.  
> También conserva el `mobility_share` y el `powerbi_recipient`; únicamente intenta retirar del Share las tablas Golden antes de eliminarlas.


In [0]:
%python
dbutils.widgets.removeAll()


In [0]:
%python
dbutils.widgets.text("storageName", "adlsproyecto")
dbutils.widgets.text("catalogo", "catalog_au")


In [0]:
%python
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

ruta_bronze = f"abfss://bronze@{storageName}.dfs.core.windows.net"
ruta_silver = f"abfss://silver@{storageName}.dfs.core.windows.net"
ruta_golden = f"abfss://golden@{storageName}.dfs.core.windows.net"

print(f"Catálogo    : {catalogo}")
print(f"Storage     : {storageName}")
print(f"Ruta Bronze : {ruta_bronze}")
print(f"Ruta Silver : {ruta_silver}")
print(f"Ruta Golden : {ruta_golden}")


### Eliminación tablas Bronze


In [0]:
%sql
-- ============================================================
-- DROP TABLES BRONZE
-- ============================================================

DROP TABLE IF EXISTS ${catalogo}.bronze.taxi_trips;
DROP TABLE IF EXISTS ${catalogo}.bronze.citibike_trips;
DROP TABLE IF EXISTS ${catalogo}.bronze.weather_hourly;
DROP TABLE IF EXISTS ${catalogo}.bronze.taxi_zones_geojson;
DROP TABLE IF EXISTS ${catalogo}.bronze.taxi_zones_csv;


In [0]:
%python
# ============================================================
# REMOVE DATA - BRONZE
# ============================================================

rutas_bronze = [
    f"{ruta_bronze}/taxi_trips/",
    f"{ruta_bronze}/citibike_trips/",
    f"{ruta_bronze}/weather_hourly/",
    f"{ruta_bronze}/taxi_zones_geojson/",
    f"{ruta_bronze}/taxi_zones_csv/"
]

for ruta in rutas_bronze:
    eliminado = dbutils.fs.rm(ruta, True)
    print(f"{'ELIMINADA' if eliminado else 'NO EXISTE'} : {ruta}")


### Eliminación tablas Silver


In [0]:
%sql
-- ============================================================
-- DROP TABLES SILVER
-- ============================================================

DROP TABLE IF EXISTS ${catalogo}.silver.taxi_trips;
DROP TABLE IF EXISTS ${catalogo}.silver.citibike_trips;
DROP TABLE IF EXISTS ${catalogo}.silver.weather_hourly;
DROP TABLE IF EXISTS ${catalogo}.silver.taxi_zone_features;
DROP TABLE IF EXISTS ${catalogo}.silver.taxi_zones;
DROP TABLE IF EXISTS ${catalogo}.silver.taxi_trips_enriched;
DROP TABLE IF EXISTS ${catalogo}.silver.citibike_trips_enriched;
DROP TABLE IF EXISTS ${catalogo}.silver.mobility_events;


In [0]:
%python
# ============================================================
# REMOVE DATA - SILVER
# ============================================================

rutas_silver = [
    f"{ruta_silver}/taxi_trips/",
    f"{ruta_silver}/citibike_trips/",
    f"{ruta_silver}/weather_hourly/",
    f"{ruta_silver}/taxi_zone_features/",
    f"{ruta_silver}/taxi_zones/",
    f"{ruta_silver}/taxi_trips_enriched/",
    f"{ruta_silver}/citibike_trips_enriched/",
    f"{ruta_silver}/mobility_events/"
]

for ruta in rutas_silver:
    eliminado = dbutils.fs.rm(ruta, True)
    print(f"{'ELIMINADA' if eliminado else 'NO EXISTE'} : {ruta}")


### Eliminación tablas Golden


In [0]:
%python
# ============================================================
# RETIRAR TABLAS GOLDEN DE DELTA SHARING
# ============================================================
# Esta sección mantiene el Share y el Recipient, pero retira
# las tablas Golden antes de eliminarlas del catálogo.
#
# Si una tabla ya no está publicada en el Share, se informa y
# se continúa con la reversión.

tablas_share = [
    f"{catalogo}.golden.mobility_hourly",
    f"{catalogo}.golden.mobility_weather",
    f"{catalogo}.golden.mobility_zone",
    f"{catalogo}.golden.transport_comparison"
]

for tabla in tablas_share:
    try:
        spark.sql(
            f"ALTER SHARE mobility_share REMOVE TABLE {tabla}"
        )
        print(f"REMOVIDA DEL SHARE : {tabla}")
    except Exception as e:
        print(
            f"NO SE REMOVIÓ DEL SHARE : {tabla} | "
            f"{str(e).splitlines()[0]}"
        )


In [0]:
%sql
-- ============================================================
-- DROP TABLES GOLDEN
-- ============================================================

DROP TABLE IF EXISTS ${catalogo}.golden.mobility_hourly;
DROP TABLE IF EXISTS ${catalogo}.golden.mobility_weather;
DROP TABLE IF EXISTS ${catalogo}.golden.mobility_zone;
DROP TABLE IF EXISTS ${catalogo}.golden.transport_comparison;


In [0]:
%python
# ============================================================
# REMOVE DATA - GOLDEN
# ============================================================

rutas_golden = [
    f"{ruta_golden}/mobility_hourly/",
    f"{ruta_golden}/mobility_weather/",
    f"{ruta_golden}/mobility_zone/",
    f"{ruta_golden}/transport_comparison/"
]

for ruta in rutas_golden:
    eliminado = dbutils.fs.rm(ruta, True)
    print(f"{'ELIMINADA' if eliminado else 'NO EXISTE'} : {ruta}")


### Validación de Reversión Medallion


In [0]:
%sql
-- ============================================================
-- VALIDAR TABLAS BRONZE
-- ============================================================

SHOW TABLES IN ${catalogo}.bronze;


In [0]:
%sql
-- ============================================================
-- VALIDAR TABLAS SILVER
-- ============================================================

SHOW TABLES IN ${catalogo}.silver;


In [0]:
%sql
-- ============================================================
-- VALIDAR TABLAS GOLDEN
-- ============================================================

SHOW TABLES IN ${catalogo}.golden;


In [0]:
%python
# ============================================================
# VALIDAR RUTAS FÍSICAS
# ============================================================

def validar_rutas(nombre_capa, rutas):
    print(f"=== {nombre_capa} ===")

    for ruta in rutas:
        try:
            dbutils.fs.ls(ruta)
            print(f"EXISTE    : {ruta}")
        except Exception:
            print(f"ELIMINADA : {ruta}")

    print()


validar_rutas("BRONZE", rutas_bronze)
validar_rutas("SILVER", rutas_silver)
validar_rutas("GOLDEN", rutas_golden)


In [0]:
%python
# ============================================================
# RESUMEN FINAL
# ============================================================

print("============================================")
print("REVERSIÓN MEDALLION FINALIZADA")
print("============================================")
print("RAW     : CONSERVADA")
print("BRONZE  : TABLAS Y RUTAS PROCESADAS")
print("SILVER  : TABLAS Y RUTAS PROCESADAS")
print("GOLDEN  : TABLAS Y RUTAS PROCESADAS")
print("CATÁLOGO / SCHEMAS / EXTERNAL LOCATIONS : CONSERVADOS")
print("DELTA SHARE / RECIPIENT                 : CONSERVADOS")
print("============================================")
